# Fine-tune SpeechBrain Tacotron 2 on a 200-Utterance Dataset (Google Colab)

This notebook adapts the Tacotron 2 + HiFi-GAN inference workflow for your friend's assignment dataset.

**Expected Google Drive layout**
```text
MyDrive/
├── datasetforassign/
│   ├── assign.01.1.wav
│   ├── assign.02.1.wav
│   └── ... (about 200 WAV files)
└── metadata file for the 200 sentences
```

The notebook:
1. Mounts Google Drive and finds audio/metadata files.
2. Inspects metadata and audio properties.
3. Matches transcripts to WAV files and lets you review the mapping.
4. Builds LJSpeech-style manifests with a reproducible 80/10/10 train/validation/test split.
5. Fine-tunes the pretrained SpeechBrain Tacotron 2 acoustic model.
6. Uses the pretrained SpeechBrain HiFi-GAN vocoder to synthesize a test sentence.

**Important:** This workflow fine-tunes Tacotron 2 and uses a pretrained HiFi-GAN vocoder; it does not fine-tune HiFi-GAN. Run cells from top to bottom with a GPU runtime. Review the transcript/audio matching before training. The metadata file path and format may need adjustment in the configuration cell.

## 1. Install dependencies and load pretrained models

In [1]:
%pip install -q speechbrain soundfile tgt Unidecode

import torch
import soundfile as sf
import pandas as pd
from pathlib import Path
from IPython.display import Audio, display
from speechbrain.inference.TTS import Tacotron2
from speechbrain.inference.vocoders import HIFIGAN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Download/cache the pretrained model files used for initialization and vocoding.
tacotron2 = Tacotron2.from_hparams(
    source="speechbrain/tts-tacotron2-ljspeech",
    savedir="/content/pretrained_tacotron2_200",
    run_opts={"device": DEVICE},
)
hifi_gan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-ljspeech",
    savedir="/content/pretrained_hifigan_200",
    run_opts={"device": DEVICE},
)

PRETRAINED_CKPT = Path("/content/pretrained_tacotron2_200/model.ckpt")
assert PRETRAINED_CKPT.is_file(), f"Pretrained checkpoint not found: {PRETRAINED_CKPT}"
print("Pretrained Tacotron 2 checkpoint:", PRETRAINED_CKPT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.7/912.7 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 39.4 MB/s eta 0:00:00


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-tacotron2-ljspeech' if not cached


Device: cuda


hyperparams.yaml:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch model.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-tacotron2-ljspeech' if not cached


...

INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-ljspeech' if not cached


generator.ckpt: reconstructing file:   0%|          |  0.00B / 55.8MB            

generator.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


Pretrained Tacotron 2 checkpoint: /content/pretrained_tacotron2_200/model.ckpt


## 2. Mount Drive and locate the 200 audio files and metadata

In [4]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import re, os, json, random, shutil, csv, subprocess, sys
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

MYDRIVE = Path("/content/drive/MyDrive")
AUDIO_DIR = MYDRIVE / "datasetforassign"

# If the metadata file is not auto-detected, set this to its exact path, e.g.
SOURCE_METADATA = MYDRIVE / "metadata_001-200.csv"


WORK_DIR = Path("/content/my_voice_tts_200")
DATA_DIR = WORK_DIR / "LJSpeech-1.1"
WAV_DIR = DATA_DIR / "wavs"
OUTPUT_DIR = WORK_DIR / "results"
PERSIST_DIR = MYDRIVE / "datasetforassign_tts_200"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

assert AUDIO_DIR.is_dir(), f"Audio input directory not found: {AUDIO_DIR}"

def natural_key(path):
    return [int(x) if x.isdigit() else x.lower()
            for x in re.split(r"(\d+)", Path(path).name)]

audio_files = sorted(AUDIO_DIR.glob("*.wav"), key=natural_key)
print("Audio directory:", AUDIO_DIR)
print("WAV count:", len(audio_files))
print("Example filenames:", [p.name for p in audio_files[:15]])
if len(audio_files) != 200:
    print(f"WARNING: expected about 200 WAVs; found {len(audio_files)}.")

# Discover likely metadata files. If multiple files appear, select the correct one manually.
metadata_candidates = [
    p for p in MYDRIVE.rglob("*")
    if p.is_file() and p.suffix.lower() in {".csv", ".tsv", ".txt"}
    and any(word in p.name.lower() for word in ["metadata", "transcript", "sentence", "assign", "text"])
]
print("\nPossible metadata files:")
for p in metadata_candidates[:50]:
    print(" ", p)

if SOURCE_METADATA is None:
    if len(metadata_candidates) == 1:
        SOURCE_METADATA = metadata_candidates[0]
        print("\nAuto-selected metadata:", SOURCE_METADATA)
    else:
        print("\nSet SOURCE_METADATA manually above if there is more than one candidate.")

if SOURCE_METADATA is not None:
    SOURCE_METADATA = Path(SOURCE_METADATA)
    assert SOURCE_METADATA.is_file(), f"Metadata file not found: {SOURCE_METADATA}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Audio directory: /content/drive/MyDrive/datasetforassign
WAV count: 200
Example filenames: ['assign.01.1.wav', 'assign.02.2.wav', 'assign.03.3.wav', 'assign.04.4.wav', 'assign.05.5.wav', 'assign.06.6.wav', 'assign.07.7.wav', 'assign.08.8.wav', 'assign.09.9.wav', 'assign.10.10.wav', 'assign.11.11.wav', 'assign.12.12.wav', 'assign.13.13.wav', 'assign.14.14.wav', 'assign.15.15.wav']

Possible metadata files:
  /content/drive/MyDrive/metadata_001-200.csv
  /content/drive/MyDrive/metadata_001-500.csv
  /content/drive/MyDrive/Punitha_FPU-20251201T075159Z-1-001/Punitha_FPU/tamil_text_200.txt


In [5]:
# ----- Configure these after checking your metadata file -----
METADATA_HAS_HEADER = False
ID_COLUMN = 0
TRANSCRIPT_COLUMN = 1
NORMALIZED_TRANSCRIPT_COLUMN = 2  # set to None if there is no normalized transcript column
USE_ROW_ORDER_FALLBACK = True     # only use this if metadata row order matches natural-sorted WAV order

assert SOURCE_METADATA is not None and Path(SOURCE_METADATA).is_file(), (
    "Set SOURCE_METADATA in the previous cell to the metadata file path."
)
assert len(audio_files) > 0, f"No WAV files found in {AUDIO_DIR}"

# Try common delimiters. Prefer a pipe-delimited file if it yields 2–3 columns;
# otherwise choose the candidate with the most plausible number of columns.
delimiter_attempts = []
for sep in ["|", ",", "\\t", ";"]:
    try:
        frame = pd.read_csv(
            SOURCE_METADATA, sep=sep, header=0 if METADATA_HAS_HEADER else None,
            dtype=str, keep_default_na=False, engine="python"
        )
        delimiter_attempts.append((frame.shape[1], sep, frame))
    except Exception as exc:
        print("Could not parse with delimiter", repr(sep), exc)
if not delimiter_attempts:
    raise ValueError("Could not parse the metadata file.")
pipe_candidate = next((item for item in delimiter_attempts if item[1] == "|" and item[0] in (2, 3)), None)
if pipe_candidate is not None:
    _, METADATA_SEPARATOR, metadata_raw = pipe_candidate
else:
    plausible = [item for item in delimiter_attempts if item[0] in (2, 3)]
    chosen = plausible[0] if plausible else max(delimiter_attempts, key=lambda item: item[0])
    _, METADATA_SEPARATOR, metadata_raw = chosen
print("Metadata path:", SOURCE_METADATA)
print("Detected delimiter:", repr(METADATA_SEPARATOR))
print("Metadata shape:", metadata_raw.shape)
display(metadata_raw.head(10))

if METADATA_HAS_HEADER:
    def get_col(column):
        return column if isinstance(column, str) else metadata_raw.columns[column]
    id_col = get_col(ID_COLUMN)
    text_col = get_col(TRANSCRIPT_COLUMN)
    norm_col = None if NORMALIZED_TRANSCRIPT_COLUMN is None else get_col(NORMALIZED_TRANSCRIPT_COLUMN)
    ids = metadata_raw[id_col].astype(str).str.strip()
    transcripts = metadata_raw[text_col].astype(str).str.strip()
    if norm_col is not None:
        normalized = metadata_raw[norm_col].astype(str).str.strip()
        transcripts = normalized.where(normalized.ne(""), transcripts)
else:
    if metadata_raw.shape[1] <= max(ID_COLUMN, TRANSCRIPT_COLUMN):
        raise ValueError("ID_COLUMN or TRANSCRIPT_COLUMN is outside the metadata columns.")
    ids = metadata_raw.iloc[:, ID_COLUMN].astype(str).str.strip()
    transcripts = metadata_raw.iloc[:, TRANSCRIPT_COLUMN].astype(str).str.strip()
    if NORMALIZED_TRANSCRIPT_COLUMN is not None and metadata_raw.shape[1] > NORMALIZED_TRANSCRIPT_COLUMN:
        normalized = metadata_raw.iloc[:, NORMALIZED_TRANSCRIPT_COLUMN].astype(str).str.strip()
        transcripts = normalized.where(normalized.ne(""), transcripts)

metadata = pd.DataFrame({"source_id": ids, "transcript": transcripts})
metadata = metadata[metadata["source_id"].str.strip().ne("")].reset_index(drop=True)
if metadata["transcript"].str.strip().eq("").any():
    bad = metadata.index[metadata["transcript"].str.strip().eq("")].tolist()[:10]
    raise ValueError(f"Empty transcript(s) found at metadata row indices: {bad}")

def id_candidates(value):
    value = str(value).strip()
    path = Path(value)
    stem = path.stem
    result = [value, path.name, stem]
    # For assign.01.1.wav, also try 01.1
    match = re.search(r"assign[._-](.+)$", stem, flags=re.IGNORECASE)
    if match:
        result.append(match.group(1))
    # Try an all-numeric/dotted token such as 01.1 or 001
    match = re.search(r"\\d+(?:\\.\\d+)*", stem)
    if match:
        result.append(match.group(0))
    # Try a normalized integer ID (001 -> 1)
    match = re.fullmatch(r"0*(\\d+)", stem)
    if match:
        result.append(str(int(match.group(1))))
    return list(dict.fromkeys(result))

audio_lookup = {}
for path in audio_files:
    for key in id_candidates(path.name) + id_candidates(path.stem):
        audio_lookup.setdefault(key, path)

matched = []
unmatched_metadata = []
used_audio = set()
for _, row in metadata.iterrows():
    found = None
    for candidate in id_candidates(row["source_id"]):
        if candidate in audio_lookup and str(audio_lookup[candidate]) not in used_audio:
            found = audio_lookup[candidate]
            break
    if found is not None:
        used_audio.add(str(found))
        matched.append({"source_id": row["source_id"], "source_wav": str(found), "transcript": row["transcript"]})
    else:
        unmatched_metadata.append(row["source_id"])

# Optional fallback: use natural-sorted WAV order for unmatched metadata rows only.
# This is safe only if the metadata row order corresponds to the sorted audio order.
if unmatched_metadata and USE_ROW_ORDER_FALLBACK and len(matched) == 0 and len(metadata) == len(audio_files):
    print("WARNING: No IDs matched exactly. Using metadata row order against natural-sorted WAVs.")

    already_used = {r["source_wav"] for r in matched}
    remaining_audio = [p for p in audio_files if str(p) not in already_used]
    unmatched_ids = set(unmatched_metadata)
    new_matched = []
    audio_iter = iter(remaining_audio)
    for _, row in metadata.iterrows():
        if row["source_id"] in unmatched_ids:
            try:
                path = next(audio_iter)
                new_matched.append({"source_id": row["source_id"], "source_wav": str(path), "transcript": row["transcript"]})
            except StopIteration:
                break
    matched.extend(new_matched)
    unmatched_metadata = []

if unmatched_metadata:
    print("Unmatched metadata IDs (first 20):", unmatched_metadata[:20])

matched_df = pd.DataFrame(matched)
if matched_df.empty:
    raise ValueError("No metadata/audio matches found. Check ID_COLUMN and the metadata format.")
if matched_df["source_wav"].duplicated().any():
    raise ValueError("A WAV was matched to multiple metadata rows. Review the ID mapping.")

print("Matched utterances:", len(matched_df), "| Audio files:", len(audio_files), "| Metadata rows:", len(metadata))
print("Review this mapping before training:")
display(matched_df.head(20))
if len(matched_df) != 200 or len(metadata) != 200 or len(audio_files) != 200:
    print("WARNING: expected exactly 200 matched audio/transcript pairs. Resolve this before training.")

# Standardize filenames to avoid punctuation/ID convention problems in the recipe.
WAV_DIR.mkdir(parents=True, exist_ok=True)
standardized_rows = []
for index, row in matched_df.reset_index(drop=True).iterrows():
    utt_id = f"{index + 1:04d}"
    source_wav = Path(row["source_wav"])
    target_wav = WAV_DIR / f"{utt_id}.wav"
    if not target_wav.exists() or target_wav.stat().st_size != source_wav.stat().st_size:
        shutil.copy2(source_wav, target_wav)
    standardized_rows.append({
        "uttid": utt_id,
        "wav": str(target_wav.resolve()),
        "label": row["transcript"].strip(),
        "segment": False,
        "duration": sf.info(str(target_wav)).duration,
        "source_id": row["source_id"],
        "source_wav": str(source_wav)
    })

standardized_df = pd.DataFrame(standardized_rows)
# LJSpeech-style metadata file: ID | transcript | normalized transcript
with open(DATA_DIR / "metadata.csv", "w", encoding="utf-8") as f:
    for row in standardized_rows:
        f.write(f"{row['uttid']}|{row['label']}|{row['label']}\\n")

print("Prepared dataset:", DATA_DIR)
print("Prepared WAVs:", len(list(WAV_DIR.glob("*.wav"))))
print("Total duration (minutes):", round(sum(r["duration"] for r in standardized_rows) / 60, 2))
print("Sample-rate counts:")
print(pd.Series([sf.info(r["source_wav"]).samplerate for r in standardized_rows]).value_counts().sort_index())


Could not parse with delimiter ',' Expected 1 fields in line 29, saw 5
Metadata path: /content/drive/MyDrive/metadata_001-200.csv
Detected delimiter: '|'
Metadata shape: (200, 3)


,0,1,2
0,001,The birch canoe slid on the smooth planks.,The birch canoe slid on the smooth planks.
1,002,Glue the sheet to the dark blue background.,Glue the sheet to the dark blue background.
2,003,It's easy to tell the depth of a well.,It's easy to tell the depth of a well.
3,004,These days a chicken leg is a rare dish.,These days a chicken leg is a rare dish.
4,005,Rice is often served in round bowls.,Rice is often served in round bowls.
5,006,The juice of lemons makes fine punch.,The juice of lemons makes fine punch.
6,007,The box was thrown beside the parked truck.,The box was thrown beside the parked truck.
7,008,The hogs were fed chopped corn and garbage.,The hogs were fed chopped corn and garbage.
8,009,Four hours of steady work faced us.,Four hours of steady work faced us.
9,010,A large size in stockings is hard to sell.,A large size in stockings is hard to sell.


Unmatched metadata IDs (first 20): ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020']
Matched utterances: 101 | Audio files: 200 | Metadata rows: 200
Review this mapping before training:


,source_id,source_wav,transcript
0,100,/content/drive/MyDrive/datasetforassign/assign...,The bill was paid every third week.
1,101,/content/drive/MyDrive/datasetforassign/assign...,Oak is strong and also gives shade.
2,102,/content/drive/MyDrive/datasetforassign/assign...,Cats and dogs each hate the other.
3,103,/content/drive/MyDrive/datasetforassign/assign...,The pipe began to rust while new.
4,104,/content/drive/MyDrive/datasetforassign/assign...,Open the crate but don't break the glass.
5,105,/content/drive/MyDrive/datasetforassign/assign...,Add the sum to the product of these three.
6,106,/content/drive/MyDrive/datasetforassign/assign...,Thieves who rob friends deserve jail.
7,107,/content/drive/MyDrive/datasetforassign/assign...,The ripe taste of cheese improves with age.
8,108,/content/drive/MyDrive/datasetforassign/assign...,Act on these orders with great speed.
9,109,/content/drive/MyDrive/datasetforassign/assign...,The hog crawled under the high fence.


Prepared dataset: /content/my_voice_tts_200/LJSpeech-1.1
Prepared WAVs: 101
Total duration (minutes): 5.98
Sample-rate counts:
22050    101
Name: count, dtype: int64


## 4. Get the SpeechBrain recipe and prepare the fine-tuning script

In [6]:
import subprocess
import sys

REPO = Path("/content/speechbrain_repo")
REPO_URL = "https://github.com/speechbrain/speechbrain.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO)],
        check=True,
    )

RECIPE_DIR = REPO / "recipes/LJSpeech/TTS/tacotron2"
TRAIN_SCRIPT = RECIPE_DIR / "train.py"
HYPERPARAMS = RECIPE_DIR / "hparams/train.yaml"
CUSTOM_TRAIN_SCRIPT = WORK_DIR / "train_finetune.py"
PREP_HELPER = RECIPE_DIR / "ljspeech_prepare.py"

assert TRAIN_SCRIPT.is_file(), f"Training recipe missing: {TRAIN_SCRIPT}"
assert HYPERPARAMS.is_file(), f"Hyperparameters missing: {HYPERPARAMS}"
assert PREP_HELPER.is_file(), f"Dataset preparation helper missing: {PREP_HELPER}"

# Keep the repository's original train.py untouched; patch a separate copy
# so it initializes the recipe model with the pretrained Tacotron 2 weights.
source = TRAIN_SCRIPT.read_text(encoding="utf-8")
needle = "        hparams = load_hyperpyyaml(fin, overrides)"
insertion = '''        hparams = load_hyperpyyaml(fin, overrides)

    # Initialize Tacotron 2 with the pretrained SpeechBrain weights.
    pretrained_path = "/content/pretrained_tacotron2_200/model.ckpt"
    pretrained_state = torch.load(
        pretrained_path, map_location="cpu", weights_only=True
    )
    result = hparams["model"].load_state_dict(pretrained_state, strict=True)
    print("Pretrained Tacotron 2 weights loaded:", result)
'''

if "Initialize Tacotron 2 with the pretrained SpeechBrain weights." not in source:
    if source.count(needle) != 1:
        raise RuntimeError("Could not safely find the model initialization point in train.py.")
    source = source.replace(needle, insertion, 1)

# The recipe imports its helper by module name; add the recipe directory to sys.path.
path_setup = f"import sys\nsys.path.insert(0, {str(RECIPE_DIR)!r})\n"
if not source.startswith(path_setup):
    source = path_setup + source

CUSTOM_TRAIN_SCRIPT.write_text(source, encoding="utf-8")

print("Custom training script:", CUSTOM_TRAIN_SCRIPT)
print("Recipe config:", HYPERPARAMS)


Custom training script: /content/my_voice_tts_200/train_finetune.py
Recipe config: /content/speechbrain_repo/recipes/LJSpeech/TTS/tacotron2/hparams/train.yaml


## 5. Create train/validation/test manifests (80/10/10)

In [12]:

from pathlib import Path

AUDIO_DIR = Path("/content/drive/MyDrive/datasetforassign")

wav_files = list(AUDIO_DIR.glob("*.wav"))

print("WAV files in source folder:", len(wav_files))
print("Rows in standardized_rows:", len(standardized_rows))

print("\nFirst 10 source filenames:")
for p in sorted(wav_files)[:10]:
    print(p.name)

print("\nFirst 10 standardized IDs:")
for row in standardized_rows[:10]:
    print(row["uttid"], "->", row["wav"])


WAV files in source folder: 200
Rows in standardized_rows: 101

First 10 source filenames:
assign.01.1.wav
assign.02.2.wav
assign.03.3.wav
assign.04.4.wav
assign.05.5.wav
assign.06.6.wav
assign.07.7.wav
assign.08.8.wav
assign.09.9.wav
assign.10.10.wav

First 10 standardized IDs:
0001 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0001.wav
0002 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0002.wav
0003 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0003.wav
0004 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0004.wav
0005 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0005.wav
0006 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0006.wav
0007 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0007.wav
0008 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0008.wav
0009 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0009.wav
0010 -> /content/my_voice_tts_200/LJSpeech-1.1/wavs/0010.wav


In [13]:

from pathlib import Path
import csv
import shutil
import soundfile as sf

AUDIO_DIR = Path("/content/drive/MyDrive/datasetforassign")

# IMPORTANT: Change this if your metadata file is stored elsewhere.
SOURCE_METADATA = Path("/content/drive/MyDrive/metadata_001-200.csv")

# Reuse the standardized LJSpeech-style folder.
WAV_OUT = OUTPUT_DIR / "LJSpeech-1.1" / "wavs"
WAV_OUT.mkdir(parents=True, exist_ok=True)

# Read metadata
metadata = []
with open(SOURCE_METADATA, "r", encoding="utf-8-sig") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("|")
        if len(parts) >= 2:
            metadata.append({
                "id": parts[0].strip(),
                "text": parts[1].strip()
            })

print("Metadata rows:", len(metadata))

# Index audio files by their numeric filename component.
wav_files = sorted(AUDIO_DIR.glob("*.wav"))
audio_by_number = {}

for wav in wav_files:
    # assign.01.1.wav -> 1; assign.10.10.wav -> 10
    parts = wav.stem.split(".")
    if len(parts) >= 2 and parts[1].isdigit():
        audio_by_number[int(parts[1])] = wav

print("Audio files indexed:", len(audio_by_number))

standardized_rows = []
unmatched_metadata = []

for i, item in enumerate(metadata, start=1):
    # Try numeric ID first; use row order only as a fallback if needed.
    digits = "".join(ch for ch in item["id"] if ch.isdigit())
    number = int(digits) if digits else i

    source_wav = audio_by_number.get(number)

    if source_wav is None:
        unmatched_metadata.append(item["id"])
        continue

    uttid = f"{i:04d}"
    target_wav = WAV_OUT / f"{uttid}.wav"

    shutil.copy2(source_wav, target_wav)

    info = sf.info(str(target_wav))
    standardized_rows.append({
        "uttid": uttid,
        "wav": str(target_wav),
        "label": item["text"],
        "duration": info.duration
    })

print("Matched rows:", len(standardized_rows))
print("Unmatched metadata IDs:", unmatched_metadata[:20])


Metadata rows: 200
Audio files indexed: 200
Matched rows: 200
Unmatched metadata IDs: []


In [14]:

import json
import random

SAVE_DIR = OUTPUT_DIR / "save"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

assert len(standardized_rows) == 200, (
    f"Expected 200 rows, got {len(standardized_rows)}"
)

records = []
for row in standardized_rows:
    assert row["label"].strip(), f"Empty transcript: {row['uttid']}"
    assert Path(row["wav"]).exists(), f"Missing WAV: {row['wav']}"

    records.append({
        "uttid": row["uttid"],
        "wav": row["wav"],
        "label": row["label"].strip(),
        "segment": False,
        "duration": row["duration"],
    })

# Reproducible shuffle and split
random.Random(1234).shuffle(records)

train_records = records[:160]
valid_records = records[160:180]
test_records = records[180:200]

def write_manifest(filename, subset):
    path = SAVE_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            {r["uttid"]: r for r in subset},
            f,
            ensure_ascii=False,
            indent=2
        )
    return path

train_path = write_manifest("train.json", train_records)
valid_path = write_manifest("valid.json", valid_records)
test_path = write_manifest("test.json", test_records)

print("Train:", len(train_records))
print("Validation:", len(valid_records))
print("Test:", len(test_records))
print("Manifests saved to:", SAVE_DIR)


Train: 160
Validation: 20
Test: 20
Manifests saved to: /content/my_voice_tts_200/results/save


In [8]:
from pathlib import Path
import re

AUDIO_DIR = Path("/content/drive/MyDrive/datasetforassign")

wav_files = sorted(AUDIO_DIR.glob("*.wav"))

print("Total WAV files:", len(wav_files))
print("First 15 filenames:")
for p in wav_files[:15]:
    print(" ", p.name)

print("\nLast 15 filenames:")
for p in wav_files[-15:]:
    print(" ", p.name)

Total WAV files: 200
First 15 filenames:
  assign.01.1.wav
  assign.02.2.wav
  assign.03.3.wav
  assign.04.4.wav
  assign.05.5.wav
  assign.06.6.wav
  assign.07.7.wav
  assign.08.8.wav
  assign.09.9.wav
  assign.10.10.wav
  assign.100.100.wav
  assign.101.101.wav
  assign.102.102.wav
  assign.103.103.wav
  assign.104.104.wav

Last 15 filenames:
  assign.85.85.wav
  assign.86.86.wav
  assign.87.87.wav
  assign.88.88.wav
  assign.89.89.wav
  assign.90.90.wav
  assign.91.91.wav
  assign.92.92.wav
  assign.93.93.wav
  assign.94.94.wav
  assign.95.95.wav
  assign.96.96.wav
  assign.97.97.wav
  assign.98.98.wav
  assign.99.99.wav


In [9]:
import csv
import re
from pathlib import Path

AUDIO_DIR = Path("/content/drive/MyDrive/datasetforassign")
SOURCE_METADATA = Path("/content/drive/MyDrive/metadata_001-200.csv")

# Map each audio file by its utterance number.
wav_by_id = {}

for wav_path in AUDIO_DIR.glob("*.wav"):
    match = re.search(r"assign\.(\d+)\.\d+\.wav$", wav_path.name, re.I)

    if match:
        utt_id = int(match.group(1))
        if utt_id in wav_by_id:
            print("Duplicate audio ID:", utt_id, wav_by_id[utt_id].name, wav_path.name)
        else:
            wav_by_id[utt_id] = wav_path

# Metadata format: ID | transcript | normalized transcript
records = []
metadata_ids = set()

with open(SOURCE_METADATA, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter="|")

    for row in reader:
        if len(row) < 2:
            continue

        try:
            utt_id = int(row[0].strip())
        except ValueError:
            continue  # Skip a header row, if present.

        transcript = row[1].strip()
        if not transcript:
            continue

        metadata_ids.add(utt_id)

        if utt_id in wav_by_id:
            records.append({
                "id": f"{utt_id:03d}",
                "wav": str(wav_by_id[utt_id]),
                "text": transcript,
            })

matched_ids = {int(r["id"]) for r in records}
missing_audio = sorted(metadata_ids - set(wav_by_id))
missing_metadata = sorted(set(wav_by_id) - metadata_ids)

print("Audio files identified:", len(wav_by_id))
print("Metadata IDs:", len(metadata_ids))
print("Matched records:", len(records))
print("Metadata IDs with no audio:", missing_audio[:30])
print("Audio IDs with no metadata:", missing_metadata[:30])

if len(records) != 200:
    raise RuntimeError(
        f"Still found {len(records)} matches, not 200. "
        "Check the printed filenames and unmatched IDs before proceeding."
    )

print("Successfully matched all 200 records.")

Audio files identified: 200
Metadata IDs: 200
Matched records: 200
Metadata IDs with no audio: []
Audio IDs with no metadata: []
Successfully matched all 200 records.


## 6. Fine-tune Tacotron 2

In [15]:
# This is the training run. It uses the pretrained checkpoint injected above.
# skip_prep=True preserves the verified manifests created in the previous cell.

command = [
    sys.executable,
    str(CUSTOM_TRAIN_SCRIPT),
    str(HYPERPARAMS),
    "--device=" + ("cuda:0" if torch.cuda.is_available() else "cpu"),
    "--max_grad_norm=1.0",
    f"--data_folder={DATA_DIR}",
    f"--output_folder={OUTPUT_DIR}",
    "--skip_prep=True",
    "--epochs=15",
    "--learning_rate=0.00005",
    "--batch_size=4",
]

print("Starting fine-tuning...")
print(f"Training examples: {len(train_records)} | Validation examples: {len(valid_records)} | Test examples: {len(test_records)}")
print("Epochs: 15 | Batch size: 4 | Learning rate: 0.00005")

result = subprocess.run(
    command,
    cwd=str(RECIPE_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout[-20000:])
print("Exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Training failed. Review the training log above.")
print("Training process completed.")

# Persist training outputs to Google Drive so they survive a Colab runtime reset.
import shutil
if PERSIST_DIR.exists():
    shutil.copytree(OUTPUT_DIR, PERSIST_DIR / "results", dirs_exist_ok=True)
print("Persistent results copied to:", PERSIST_DIR)


Starting fine-tuning...
Training examples: 160 | Validation examples: 20 | Test examples: 20
Epochs: 15 | Batch size: 4 | Learning rate: 0.00005
0 [00:41<00:02,  1.06s/it, train_loss=0.64]
100%|██████████| 40/40 [00:43<00:00,  1.09s/it, train_loss=0.639]

100%|██████████| 5/5 [00:02<00:00,  2.17it/s]
speechbrain.utils.train_logger - Epoch: 12, lr: 5.00e-05 - train loss: 6.21e-01, train mel_loss: 5.97e-01, train gate_loss: 9.44e-03, train attn_loss: 1.49e-02, train attn_weight: 21.76 - valid loss: 7.15e-01, valid mel_loss: 6.47e-01, valid gate_loss: 2.66e-02, valid attn_loss: 4.19e-02, valid attn_weight: 21.76
speechbrain.utils.checkpoints - Saved an end-of-epoch checkpoint in /content/my_voice_tts_200/results/save/CKPT+2026-10-21+18-15-38+00
speechbrain.utils.checkpoints - Deleted checkpoint in /content/my_voice_tts_200/results/save/CKPT+2026-10-21+18-14-50+00
speechbrain.utils.epoch_loop - Going into epoch 13

100%|██████████| 40/40 [00:44<00:00,  1.10s/it, train_loss=0.63]

100%|████

## 7. Load the fine-tuned checkpoint and synthesize a test sentence

In [16]:
# Find the most recent SpeechBrain checkpoint created by training.
checkpoint_dirs = sorted(
    [p for p in SAVE_DIR.glob("CKPT*") if (p / "model.ckpt").is_file()],
    key=lambda p: p.name,
)
if not checkpoint_dirs:
    raise FileNotFoundError(f"No fine-tuned model checkpoint found under {SAVE_DIR}")

FINETUNED_CKPT = checkpoint_dirs[-1] / "model.ckpt"
print("Using fine-tuned checkpoint:", FINETUNED_CKPT)

state = torch.load(FINETUNED_CKPT, map_location="cpu", weights_only=True)
load_result = tacotron2.mods["model"].load_state_dict(state, strict=True)
print("Checkpoint loading:", load_result)
tacotron2.eval()

test_text = (
    "Hello, this is a test of my personalized text to speech system. "
    "I am checking how naturally it reproduces my voice."
)

with torch.inference_mode():
    mel_output, mel_length, alignment = tacotron2.encode_text(test_text)
    waveform = hifi_gan.decode_batch(mel_output)

audio = waveform[0].detach().cpu().squeeze().numpy()
OUTPUT_WAV = PERSIST_DIR / "fine_tuned_test.wav"
sf.write(str(OUTPUT_WAV), audio, 22050)

print("Generated WAV:", OUTPUT_WAV)
print("Duration:", round(len(audio) / 22050, 2), "seconds")
display(Audio(audio, rate=22050))

# Copy the fine-tuned checkpoints to Drive.
shutil.copytree(SAVE_DIR, PERSIST_DIR / "save", dirs_exist_ok=True)
print("Checkpoint folder on Drive:", PERSIST_DIR / "save")


Using fine-tuned checkpoint: /content/my_voice_tts_200/results/save/CKPT+2026-10-21+18-18-03+00/model.ckpt
Checkpoint loading: <All keys matched successfully>
Generated WAV: /content/drive/MyDrive/datasetforassign_tts_200/fine_tuned_test.wav
Duration: 11.73 seconds


Checkpoint folder on Drive: /content/drive/MyDrive/datasetforassign_tts_200/save


## Notes and troubleshooting

- Output folder on Google Drive: `MyDrive/datasetforassign_tts_200/`.
- The notebook creates 160 training, 20 validation, and 20 test utterances when all 200 files are matched.
- The SpeechBrain recipe trains with the training and validation manifests; the test manifest is held aside for later evaluation.
- This workflow fine-tunes Tacotron 2 using a pretrained checkpoint and uses the pretrained HiFi-GAN vocoder for waveform generation. HiFi-GAN itself is not fine-tuned here.
- Confirm the actual audio sample rate and ensure it is compatible with the selected Tacotron 2/HiFi-GAN configuration. Do not assume the input audio is 22,050 Hz.
- Carefully verify the printed metadata-to-audio mapping, especially because filenames such as `assign.01.1.wav` may not match metadata IDs directly. If row-order fallback is used, it assumes metadata order corresponds to the natural-sorted audio order.
- For best practice, keep the test split out of training and checkpoint selection.
- The fine-tuned checkpoint and generated test WAV are copied to Google Drive so they can survive a Colab runtime reset.
- Colab needs internet access to install dependencies, clone SpeechBrain, and download pretrained checkpoints.